In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

In [2]:
# 欄位處理 Column processing
def check_data_format(df_data, label_value):
    
    # 生活習慣&疾病史，空值刪除
    df_data = df_data.dropna(subset=["AGE","DRK","SMK_EXPERIENCE","NUT_EXPERIENCE","SPO_HABIT","ORTHOPEDICS_ARTICULUS","OSTEOPOROSIS","ARTHRITIS","GOUT","LUNG_RESPIRATORY","ASTHMA","EMPHYSEMA_BRONCHITIS","HEART_BLOOD_TUBE","VALVE_HEART_DIS","CORONARY_ARTERY_DIS","ARRHYTHMIA","CARDIOMYOPATHY","CONGENITAL_HEART_DIS","OTHER_HEART_DIS","HYPERLIPIDEMIA","HYPERTENSION","APOPLEXIA","DIABETES","ALIMENTARY_CANAL","PEPTIC_ULCER","GASTROESOPHAGEA_REFLUX","IRRITABLE_BOWEL_SND","PSYCHOSIS","DEPRESSION","MANIC_DEPRESSION","POSTPARTUM_DEPRESSION","OBSESSIVE_COMPULSIVE_DIS","ALCOHOLISM_DRUG_ABUSE","SCHIZOPHRENIA","NERVOUS_SYSTEM","EPILEPSY","HEMICRANIA","MULTIPLE_SCLEROSIS","PARKISON","DEMENTIA","LIVER_GALL_STONE","KIDNEY_STONE","RENAL_FAILURE","VERTIGO","CARCINOMA_IN_SITU","LIVER_CA","LUNG_CA","BREAST_CA","GASTRIC_CA","COLORECTAL_CA","NASOPHARYNGEAL_CA","PROSTATE_CA","OTHER_CA","CATARACT","GLAUCOMA","XEROPHTHALMIA","RENTINAL_DETACHMENT","FLOATERS","BLIND","COLOR_BLIND"])
    #print("刪除生活習慣及疾病史空值後剩餘:" + str(df_data.shape))
    
    # 身體及血尿液量測數值，空值刪除
    df_data = df_data.dropna(subset=["BODY_HEIGHT","BODY_WEIGHT","BMI","BODY_WAISTLINE","BODY_BUTTOCKS","WHR","SIT_1_SYSTOLIC_PRESSURE","SIT_1_DIASTOLIC_PRESSURE","SIT_2_SYSTOLIC_PRESSURE","SIT_2_DIASTOLIC_PRESSURE","SIT_1_HEARTBEAT_SPEED","SIT_2_HEARTBEAT_SPEED","BONE_EXAM_RESULT","T_SCORE","Z_SCORE","BODY_FAT_RATE","RBC","WBC","PLATELET","HB","HCT","HBA1C","ANTI_HCV_AB_1","HBSAG_1","HBEAG_1","ANTI_HBS_AB_1","ANTI_HBC_AB_1","FASTING_GLUCOSE","T_CHO","TG","HDL_C","LDL_C","T_BILIRUBIN","ALBUMIN","SGOT","SGPT","GAMMA_GT","AFP","BUN","CREATININE","URIC_ACID","MICROALB"])
    #print("刪除身體及血尿液量測數值空值後剩餘:" + str(df_data.shape))

    # 女性相關問題之類別資料，空值刪除
    df_data = df_data.dropna(subset=["MENSTRURATION_REGULAR","MENSTRURATION_CURR","PREGNANCY","SEVERE_DYSMENORRHEA","MYOMA","OVARIAN_CYST","ENDOMETRIOSIS","CERVICAL_POLYP","UTERINE_CA","CERVICAL_CA","OVARIAN_CA"])
    #print("刪除女性相關問題之類別資料空值後剩餘:" + str(df_data.shape))

    # 女性相關問題:初經年齡為空值，使用眾數取代
    #MENARCHE_cl = ctl_df.groupby('MENARCHE')
    #print(MENARCHE_cl.size())
    #'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead
    df_data['MENARCHE'] = df_data['MENARCHE'].fillna(df_data['MENARCHE'].mode()[0])
    #df.fillna({'MENARCHE': df_data['MENARCHE'].mode()[0]}, inplace=True)
    #print("初經年齡眾數:" + str(df_data['MENARCHE'].mode()[0]))

    # 女性相關問題:停經年齡，已停經且停經年齡為空值，使用眾數取代
    #MENOPAUSE_cl = ctl_df.groupby('MENOPAUSE')
    #print(MENOPAUSE_cl.size())
    MENOPAUSE_mode = df_data['MENOPAUSE'].mode()[0]
    df_data['MENOPAUSE'] = df_data.apply(
        lambda row: MENOPAUSE_mode if row['MENSTRURATION_CURR'] == '1' and pd.isna(row['MENOPAUSE']) else row['MENOPAUSE'],
        axis=1
    )
    #print("停經年齡眾數:" + str(MENOPAUSE_mode))

    # 女性相關問題:新增欄位"過去及至今月經期間"，(1)已停經:停經年齡-初經年齡(2)未停經:現在年齡-初經年齡
    df_data.insert(len(df_data.columns), "MC_DURATION",True)
    df_data['MC_DURATION'] = df_data.apply(
        lambda row: int(row['MENOPAUSE'])-int(row['MENARCHE']) if row['MENSTRURATION_CURR'] == '1' else int(row['AGE'])-int(row['MENARCHE']),
        axis=1
    )

    # 女性相關問題:未曾懷孕的樣本中，"是否曾經生產過"、"生產次數"、"是否曾經流產過"、"是否曾經餵過母乳"會顯示空值，需補值
    df_data['BIRTH'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BIRTH']) else row['BIRTH'],
        axis=1
    )
    df_data['BIRTH_TIMES'] = df_data.apply(
        lambda row: '0' if row['BIRTH'] == '1' and pd.isna(row['BIRTH_TIMES']) else row['BIRTH_TIMES'],
        axis=1
    )

    df_data['ABORTION'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['ABORTION']) else row['ABORTION'],
        axis=1
    )
    df_data['BREAST_MILK'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BREAST_MILK']) else row['BREAST_MILK'],
        axis=1
    )

    # 女性相關問題:"是否曾經生產過"補值後依然有空值，刪除樣本(例如:曾經懷孕，"是否曾經生產過"卻顯示空值)
    df_data = df_data.dropna(subset=["BIRTH","BIRTH_TIMES","ABORTION","BREAST_MILK"])

    # 女性相關問題:新增欄位"是否高齡產"
    # 未曾懷孕的樣本中，"最後生產年齡"為空值，補值[-1]，協助判斷未填入"最後生產年齡"的樣本以便刪除
    df_data['BIRTH_AGE_FINAL'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BIRTH_AGE_FINAL']) else row['BIRTH_AGE_FINAL'],
        axis=1
    )
    df_data = df_data.dropna(subset=["BIRTH_AGE_FINAL"])
    #print("刪除女性相關問題無法補值之樣本後剩餘:" + str(df_data.shape))

    # 依"最後生產年齡"給值，>=35是高齡產，<35則否
    df_data.insert(69, "ELDERL_PRIMIGRAVIDA",True)
    df_data['ELDERL_PRIMIGRAVIDA'] = df_data.apply(
        lambda row: '1' if int(row['BIRTH_AGE_FINAL']) < 35 else '2',
        axis=1
    )

    # 身體及血尿液量測數值，特殊值處理
    # 平均收縮壓、舒張壓、心跳
    df_data.insert(91, "SIT_SYSTOLIC_PRESSURE_AVERAGE",True)
    df_data['SIT_SYSTOLIC_PRESSURE_AVERAGE'] = (df_data['SIT_1_SYSTOLIC_PRESSURE'].astype(float) + df_data['SIT_2_SYSTOLIC_PRESSURE'].astype(float))/2

    df_data.insert(92, "SIT_DIASTOLIC_PRESSURE_AVERAGE",True)
    df_data['SIT_DIASTOLIC_PRESSURE_AVERAGE'] = (df_data['SIT_1_DIASTOLIC_PRESSURE'].astype(float) + df_data['SIT_2_DIASTOLIC_PRESSURE'].astype(float))/2

    df_data.insert(93, "SIT_HEARTBEAT_SPEED_AVERAGE",True)
    df_data['SIT_HEARTBEAT_SPEED_AVERAGE'] = (df_data['SIT_1_HEARTBEAT_SPEED'].astype(float) + df_data['SIT_2_HEARTBEAT_SPEED'].astype(float))/2

    # 肺功能測量結果，文字代換成數字
    df_data['INTERPRETATION'] = df_data['INTERPRETATION'].map({'Normal Spirometry':1, 'Restrictive Impairment':2, 'Obstructive Impairment':3, 'Mixed Impairment':4})

    # 醣化血色素值，<3.8用0取代
    df_data['HBA1C'] = df_data['HBA1C'].replace(to_replace={'<3.8': 0})

    # C型肝炎抗體、B型肝炎表面抗原、B型肝炎e抗原、B型肝炎表面抗體、B型肝炎核心抗體，文字代換成數字
    df_data['ANTI_HCV_AB_1'] = df_data['ANTI_HCV_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['HBSAG_1'] = df_data['HBSAG_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['HBEAG_1'] = df_data['HBEAG_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['ANTI_HBS_AB_1'] = df_data['ANTI_HBS_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['ANTI_HBC_AB_1'] = df_data['ANTI_HBC_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})

    # 血清麩胺酸苯醋酸轉氨基酶，<6用0取代
    df_data['SGOT'] = df_data['SGOT'].replace(to_replace={'<6': 0})
    
    # 血清麩胺酸: 0}，<5用0取代
    df_data['SGPT'] = df_data['SGPT'].replace(to_replace={'<5': 0})

    # γ －麩胺醯轉移酶，<3.3用0取代
    df_data['GAMMA_GT'] = df_data['GAMMA_GT'].replace(to_replace={'<3.3': 0})

    # 甲型胎兒血清蛋白，<2用0取代
    df_data['AFP'] = df_data['AFP'].replace(to_replace={'< 2.00': 0, '<2': 0, '<2.0': 0, '< 2.0': 0})
    
    # 尿酸，<0.6用0取代
    df_data['URIC_ACID'] = df_data['URIC_ACID'].replace(to_replace={'<0.6': 0})

    # 尿中微蛋白，<2.09用0取代
    df_data['MICROALB'] = df_data['MICROALB'].replace(to_replace={'<2.09': 0})
    
    # 新增label
    df_data.insert(128, "OS",True)
    df_data['OS'] = label_value
    
    #print("最終樣本數:" + str(df_data.shape))

    return df_data

In [3]:
df_n = pd.read_csv ('./data/oste_normal(20240409).csv', low_memory=False)
ctl_df = df_n[1:]
print("無骨鬆原始資料:" + str(ctl_df.shape))
#ctl_df_filter = ctl_df[pd.to_numeric(ctl_df['T_SCORE']) > 2.3]
#print(ctl_df_filter.shape)

re_ctl_df = check_data_format(ctl_df, '0')
print("無骨鬆欄位處理後資料:" + str(re_ctl_df.shape))

df_oa = pd.read_csv ('./data/oste_osteopenia(20240409).csv', low_memory=False)
oa_df = df_oa[1:]
print("輕微骨鬆原始資料:" + str(oa_df.shape))
#oa_df_filter = oa_df[(pd.to_numeric(oa_df['T_SCORE']) > -2.0) & (pd.to_numeric(oa_df['T_SCORE']) < -1.5)]
#print(oa_df_filter.shape)

re_oa_df = check_data_format(oa_df, '2')
print("輕微骨鬆欄位處理後資料:" + str(re_oa_df.shape))

df_os = pd.read_csv ('./data/oste_osteoporosis(20240409).csv', low_memory=False)
os_df = df_os[1:]
print("骨鬆原始資料:" + str(os_df.shape))

re_os_df = check_data_format(os_df, '1')
print("骨鬆欄位處理後資料:" + str(re_os_df.shape))

無骨鬆原始資料:(67283, 123)
無骨鬆欄位處理後資料:(59265, 129)
輕微骨鬆原始資料:(26366, 123)
輕微骨鬆欄位處理後資料:(23027, 129)
骨鬆原始資料:(7910, 123)
骨鬆欄位處理後資料:(6703, 129)


In [4]:
# 抽取有snp資訊的樣本
snp_ctl_df = re_ctl_df.dropna(subset=["TWB2_ID"])
print(str(snp_ctl_df.shape))

snp_oa_df = re_oa_df.dropna(subset=["TWB2_ID"])
print(str(snp_oa_df.shape))

snp_os_df = re_os_df.dropna(subset=["TWB2_ID"])
print(str(snp_os_df.shape))

(43232, 129)
(17352, 129)
(4978, 129)


In [5]:
# 取得候選SNP集合可對應到TWB資料庫的SNP_ID資訊
map_file = './data/tpmi_snp.map'
snp_name_list = []
map_f = open(map_file, 'r')
for line in map_f:
    line = line.strip()
    line = line.split('\t')
    snp_id = line[1]
    snp_name_list.append(snp_id)
print("SNP List:" + str(len(snp_name_list)))
print(snp_name_list)

SNP List:9
['rs35645182', 'rs34782958', 'rs2198891', 'rs190752175', 'rs4712614', 'rs35257346', 'rs1871859', 'rs635965', 'rs867023908']


In [6]:
# 新增SNP欄位到三組dataframe
for rs_id in snp_name_list:
    snp_os_df.insert(len(snp_os_df.columns), rs_id, None)
    snp_oa_df.insert(len(snp_oa_df.columns), rs_id, None)
    snp_ctl_df.insert(len(snp_ctl_df.columns), rs_id, None)

In [7]:
# 將TWB2_ID欄位設為index
snp_os_df = snp_os_df.set_index('TWB2_ID')
snp_oa_df = snp_oa_df.set_index('TWB2_ID')
snp_ctl_df = snp_ctl_df.set_index('TWB2_ID')

In [8]:
from collections import deque
def asign_snp_allele(ped_file_name, snp_df):
    ped_f = open(ped_file_name, 'r')
    for line in ped_f:
        line = line.strip()
        line = line.split('\t')
        idx = line[0]
        #print(idx)
        snp_count = len(line[6:])/2
        snp_list = line[6:]
        #print(snp_list)
        queue = deque(snp_list)
        for i in range(int(len(snp_list)/2)):
            chr_1 = queue.popleft()
            chr_2 = queue.popleft()
            #print(chr_1)
            #print(chr_2)
            snp_name = snp_name_list[i]
            #print(snp_name)
            
            #snp_df.loc[idx, snp_name] = 
            if idx in snp_df.index:
                snp_df.at[idx, snp_name] = chr_1+'_'+chr_2
                #snp_df.at[idx, snp_name] = index_nt_biallele_code[nt_biallele_code[t_chr]]
    return snp_df

In [9]:
snp_os_df = asign_snp_allele('./data/tpmi_osteoporosis.ped', snp_os_df)
#print(snp_os_df)
snp_oa_df = asign_snp_allele('./data/tpmi_osteopenia.ped', snp_oa_df)
#print(snp_oa_df)
snp_ctl_df = asign_snp_allele('./data/tpmi_normal.ped', snp_ctl_df)
#print(snp_ctl_df)

In [10]:
# 讀取PRS file
snp_prs_df = pd.read_csv('./prs/tpmi_osteoporosis_prs.txt', sep='\t')
#print(snp_prs_df)
snp_prs_list = snp_prs_df['source'].unique()
print(snp_prs_list)

<StringArray>
['TPMI_Osteoporosis_osteopenia and pathological fracture',
                                      'TPMI_Osteoporosis',
                                  'TPMI_Osteoporosis NOS',
                               'TPMI_Senile osteoporosis']
Length: 4, dtype: str


In [11]:
# 新增PRS欄位到三組dataframe
for rs_group in snp_prs_list:    
    snp_os_df.insert(len(snp_os_df.columns), rs_group, None)
    snp_oa_df.insert(len(snp_oa_df.columns), rs_group, None)
    snp_ctl_df.insert(len(snp_ctl_df.columns), rs_group, None)

In [12]:
def alt_conut(genotype, alt):
    if pd.isna(genotype):
        return 0
    a1, a2 = genotype.split("_")
    return (a1 == alt) + (a2 == alt)
    
def col_prs(snp_df):
    for source, gdf in snp_prs_df.groupby("source"):
    #for source, gdf in snp_df.groupby("source"):
        #print(source)
        #print(gdf)
        prs = pd.Series(0.0, index=snp_df.index)

        for _, row in gdf.iterrows():
            rs = row["rs_id"]
            alt = row["ALT_allele(A1)"]
            es = row["effect_size"]
            #print(rs)
            #print(alt)
            #print(es)
            if rs not in snp_df.columns:
                #print("no_rsid")
                #print(rs)
                continue
            #print(snp_df[rs])
            count = snp_df[rs].apply(alt_conut, alt=alt)
            #print(count)
            prs += count * es
            #print(prs)
        #print(source)
        snp_df[source] = prs
        #print(snp_df[source])
    return snp_df

In [13]:
snp_os_df = col_prs(snp_os_df)
print(str(snp_os_df.shape))
snp_oa_df = col_prs(snp_oa_df)
print(str(snp_oa_df.shape))
snp_ctl_df = col_prs(snp_ctl_df)
print(str(snp_ctl_df.shape))

(4978, 141)
(17352, 141)
(43232, 141)


In [14]:
#合併所有分類資料
merge_all = pd.concat([snp_os_df, snp_ctl_df])
merge_all = pd.concat([merge_all, snp_oa_df])

print(merge_all.shape)

(65562, 141)


In [15]:
#切分train data & test data
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    merge_all,
    test_size=0.2,
    random_state=42
)

print("train shape:", train_df.shape)
print(train_df['OS'].value_counts())
print("test shape:", test_df.shape)
print(test_df['OS'].value_counts())

train shape: (52449, 141)
OS
0    34544
2    13898
1     4007
Name: count, dtype: int64
test shape: (13113, 141)
OS
0    8688
2    3454
1     971
Name: count, dtype: int64


In [16]:
def sub_feature_ectract_cate(input_df):
    #category feature select
    re_cate_df = input_df.filter(items=["MENSTRURATION_CURR"
    ], axis=1)
    return re_cate_df

def sub_feature_ectract_num(input_df):
    #numeric feature select
    re_num_df = input_df.filter(items=["AGE","FASTING_GLUCOSE","SGPT","HB","TG","HCT","BMI","BODY_WEIGHT","TPMI_Osteoporosis_osteopenia and pathological fracture"
    ], axis=1)
    return re_num_df

In [17]:
cate_train_df = sub_feature_ectract_cate(train_df)
cate_train_array = np.array(cate_train_df, dtype='float32')
num_train_df = sub_feature_ectract_num(train_df)

cate_test_df = sub_feature_ectract_cate(test_df)
cate_test_array = np.array(cate_test_df, dtype='float32')
num_test_df = sub_feature_ectract_num(test_df)


#僅使用train數值資料建立正規化規則，再套用到test數值資料，回傳資料型態為numpy array
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
num_train_array = scaler.fit_transform(num_train_df)

num_test_array = scaler.transform(num_test_df)

joblib.dump(scaler, "sub_snp_feature_oste_scaler.pkl")


['sub_snp_feature_oste_scaler.pkl']

In [18]:
#合併cate和num資料
train_array = np.concatenate([cate_train_array, num_train_array], axis=1)
train_os_array = train_df['OS'].to_numpy()

train_pos_array = train_array[train_os_array == '1']
train_neg_array = train_array[train_os_array == '0']
train_mid_array = train_array[train_os_array == '2']
print(train_pos_array.shape)
print(train_neg_array.shape)
print(train_mid_array.shape)


(4007, 10)
(34544, 10)
(13898, 10)


In [19]:
# training neg data
# 用POS取NEG

# 計算正樣本中心
median_center = np.median(train_pos_array, axis=0)
#print(median_center)
from sklearn.metrics.pairwise import euclidean_distances
# 計算負樣本到正樣本中心的距離
distances = euclidean_distances(train_neg_array, [median_center]).flatten()
#print(distances)

ext_train_neg_array = np.empty((1, train_neg_array.shape[1]), dtype=np.float32)

K = len(train_pos_array)

idx = np.argsort(distances)[-K:]
ext_train_neg_array = train_neg_array[idx]

print("POS:", train_pos_array.shape)
print("NEG:", ext_train_neg_array.shape)

POS: (4007, 10)
NEG: (4007, 10)


In [20]:
# training mid data
# 用POS和NEG取MID

ext_train_mid_array = np.empty((1, train_mid_array.shape[1]), dtype=np.float32)

# 計算正樣本中心
pos_median_center = np.median(train_pos_array, axis=0)
# 計算負樣本中心
neg_median_center = np.median(ext_train_neg_array, axis=0)

pos_distances = euclidean_distances(train_mid_array, [pos_median_center]).flatten()
neg_distances = euclidean_distances(train_mid_array, [neg_median_center]).flatten()

# 與正負樣本都保持一定距離
threshold = 2.823

mask = (
    (pos_distances >= threshold) &
    (neg_distances >= threshold)
)

ext_train_mid_array = train_mid_array[mask]

print("mid:", ext_train_mid_array.shape)

mid: (4006, 10)


In [21]:
test_array = np.concatenate([cate_test_array, num_test_array], axis=1)
test_os_array = test_df['OS'].to_numpy()

print(test_os_array)
test_pos_array = test_array[test_os_array == '1']
test_neg_array = test_array[test_os_array == '0']
test_mid_array = test_array[test_os_array == '2']
print(test_pos_array.shape)
print(test_neg_array.shape)
print(test_mid_array.shape)

['0' '1' '0' ... '0' '0' '0']
(971, 10)
(8688, 10)
(3454, 10)


In [22]:
# testing neg data
# 用POS取NEG

# 計算正樣本中心
median_center = np.median(test_pos_array, axis=0)
#print(median_center)
from sklearn.metrics.pairwise import euclidean_distances
# 計算負樣本到正樣本中心的距離
distances = euclidean_distances(test_neg_array, [median_center]).flatten()
#print(distances)

ext_test_neg_array = np.empty((1, test_neg_array.shape[1]), dtype=np.float32)

K = len(test_pos_array)

idx = np.argsort(distances)[-K:]
ext_test_neg_array = test_neg_array[idx]

print("POS:", test_pos_array.shape)
print("NEG:", ext_test_neg_array.shape)

POS: (971, 10)
NEG: (971, 10)


In [23]:
# testing mid data
# 用POS和NEG取MID

ext_test_mid_array = np.empty((1, test_mid_array.shape[1]), dtype=np.float32)

# 計算正樣本中心
pos_median_center = np.median(test_pos_array, axis=0)
# 計算負樣本中心
neg_median_center = np.median(ext_test_neg_array, axis=0)

pos_distances = euclidean_distances(test_mid_array, [pos_median_center]).flatten()
neg_distances = euclidean_distances(test_mid_array, [neg_median_center]).flatten()

# 與正負樣本都保持一定距離
threshold = 2.855

mask = (
    (pos_distances >= threshold) &
    (neg_distances >= threshold)
)

ext_test_mid_array = test_mid_array[mask]

print("mid:", ext_test_mid_array.shape)

mid: (970, 10)


In [24]:
training_X = np.vstack([train_pos_array, ext_train_neg_array])
training_X = np.vstack([training_X, ext_train_mid_array])
training_Y = np.concatenate([np.ones(train_pos_array.shape[0]), np.zeros(ext_train_neg_array.shape[0]), np.full(ext_train_mid_array.shape[0], 2)])
print(training_X.shape)
print(training_Y.shape)

testing_X = np.vstack([test_pos_array, ext_test_neg_array])
testing_X = np.vstack([testing_X, ext_test_mid_array])
testing_Y = np.concatenate([np.ones(test_pos_array.shape[0]), np.zeros(ext_test_neg_array.shape[0]), np.full(ext_test_mid_array.shape[0], 2)])
print(testing_X.shape)
print(testing_Y.shape)

(12020, 10)
(12020,)
(2912, 10)
(2912,)


In [25]:
# 儲存training&testing numpy array
np.save("sub_snp_feature_training_X.npy", training_X)
np.save("sub_snp_feature_training_Y.npy", training_Y)
np.save("sub_snp_feature_testing_X.npy", testing_X)
np.save("sub_snp_feature_testing_Y.npy", testing_Y)

In [26]:
training_X = np.load("sub_snp_feature_training_X.npy")
training_Y = np.load("sub_snp_feature_training_Y.npy")
testing_X = np.load("sub_snp_feature_testing_X.npy")
testing_Y = np.load("sub_snp_feature_testing_Y.npy")

In [27]:
import collections
print(collections.Counter(training_Y))
print(collections.Counter(testing_Y))

Counter({np.float64(1.0): 4007, np.float64(0.0): 4007, np.float64(2.0): 4006})
Counter({np.float64(1.0): 971, np.float64(0.0): 971, np.float64(2.0): 970})


In [28]:
from sklearn.metrics import accuracy_score,f1_score,matthews_corrcoef,confusion_matrix
def evalution_metrics_multi(test_label, labels_score):
    accuracy = accuracy_score(test_label, labels_score.round())
    confusion = confusion_matrix(test_label, labels_score.round())
    print(confusion)
    f1 = f1_score(test_label, labels_score.round(), average="macro")
    mcc = matthews_corrcoef(test_label, labels_score.round())
    metrics = np.round([accuracy, f1, mcc],2)
    columns=['accuracy', 'f1', 'mcc']
    metrics_dict = dict(zip(columns,metrics))
    
    return(metrics_dict)

In [29]:
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_validate
import optuna

/home/wenzhi/anaconda3/envs/twb_analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
cate_column = cate_train_df.columns
num_column = num_train_df.columns
feature_list = np.concatenate([cate_column, num_column])
print(feature_list)

['MENSTRURATION_CURR' 'AGE' 'FASTING_GLUCOSE' 'SGPT' 'HB' 'TG' 'HCT' 'BMI'
 'BODY_WEIGHT' 'TPMI_Osteoporosis_osteopenia and pathological fracture']


In [31]:
from sklearn.model_selection import KFold
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [32]:
from sklearn.inspection import permutation_importance
def get_permutation_importance(model_name, feature_list):
    result = permutation_importance(
        model_name, 
        testing_X, 
        testing_Y,
        scoring="f1_macro",
        n_repeats=30,
        random_state=42,
        n_jobs=-1
    ) 
    
    feature_importance = pd.DataFrame({
        "feature": feature_list,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std
    }).sort_values("importance_mean", ascending=False)

    print(feature_importance.head(20))

In [33]:
def svm_cv(model, X_train, Y_train, cv):
    scores = cross_validate(
        model,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            'accuracy',
            'f1_macro',
            'precision_macro',
            'recall_macro',
            'matthews_corrcoef'
        ],
        n_jobs=-1
    )
    print('Accuracy:', scores['test_accuracy'])
    print('Precision:', scores['test_precision_macro'])
    print('Recall:', scores['test_recall_macro'])
    print('F1 score', scores['test_f1_macro'])
    print('matthews_corrcoef:', scores['test_matthews_corrcoef'])

    print('Accuracy:', scores['test_accuracy'].mean())
    print('Precision:', scores['test_precision_macro'].mean())
    print('Recall:', scores['test_recall_macro'].mean())
    print('F1 score', scores['test_f1_macro'].mean())
    print('matthews_corrcoef:', scores['test_matthews_corrcoef'].mean())



In [34]:
#SVM
from sklearn import svm
import joblib
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
svc = svm.SVC(probability=True)
#svm_cv(svc, training_X, training_Y, cv)

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(svc)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_snp_feature_svc_fold_{fold}.pkl")
# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

svc_fit = svc.fit(training_X, training_Y)
svm_score = svc_fit.predict(testing_X)

print(classification_report(testing_Y.astype(float), svm_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), svm_score.astype(float))

Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.8012 ± 0.0032
precision_macro: 0.8094 ± 0.0037
recall_macro: 0.8011 ± 0.0024
f1_macro: 0.7995 ± 0.0031
matthews_corrcoef: 0.7068 ± 0.0045
              precision    recall  f1-score   support

         0.0       0.77      0.95      0.85       971
         1.0       0.94      0.75      0.84       971
         2.0       0.75      0.72      0.74       970

    accuracy                           0.81      2912
   macro avg       0.82      0.81      0.81      2912
weighted avg       0.82      0.81      0.81      2912

[[924   0  47]
 [ 57 732 182]
 [221  50 699]]


{'accuracy': np.float64(0.81), 'f1': np.float64(0.81), 'mcc': np.float64(0.72)}

In [35]:
from sklearn.ensemble import RandomForestClassifier
def rf_objective(trial, X_train, Y_train, cv):

    rf = RandomForestClassifier(
        n_estimators = trial.suggest_int("n_estimators", 10, 200),
        max_depth = trial.suggest_int("max_depth", 3, 30),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10),
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        random_state = 42,
        n_jobs = -1
    )

    scores = cross_validate(
        rf,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            'accuracy',
            'precision_macro',
            'recall_macro',
            'f1_macro',
            'matthews_corrcoef'
        ],
        n_jobs=-1
    )

    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [36]:
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: rf_objective(trial, training_X, training_Y, cv), n_trials=200)

best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_forest = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_forest)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_snp_feature_rf_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

forest_fit = best_forest.fit(training_X, training_Y)
rf_score = forest_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), rf_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), rf_score.astype(float))

[I 2026-11-18 11:49:42,507] A new study created in memory with name: no-name-f422ca55-833e-4065-bae0-3a2aa3316ace
[I 2026-11-18 11:49:45,245] Trial 0 finished with value: 0.7836560021797108 and parameters: {'n_estimators': 73, 'max_depth': 25, 'min_samples_split': 6, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7836560021797108.
[I 2026-11-18 11:49:48,212] Trial 1 finished with value: 0.7872007633637896 and parameters: {'n_estimators': 193, 'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 1 with value: 0.7872007633637896.
[I 2026-11-18 11:49:51,393] Trial 2 finished with value: 0.6830129437823212 and parameters: {'n_estimators': 189, 'max_depth': 3, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 1 with value: 0.7872007633637896.
[I 2026-11-18 11:49:54,287] Trial 3 finished with value: 0.7880960379463422 and parameters: {'n_estimators': 172, 'max_depth': 26, 'mi

{'n_estimators': 183, 'max_depth': 26, 'min_samples_split': 16, 'min_samples_leaf': 2, 'max_features': 'log2'}
Best F1 Macro: 0.7913
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7903 ± 0.0042
precision_macro: 0.8017 ± 0.0052
recall_macro: 0.7903 ± 0.0042
f1_macro: 0.7913 ± 0.0043
matthews_corrcoef: 0.6890 ± 0.0065
              precision    recall  f1-score   support

         0.0       0.80      0.88      0.84       971
         1.0       0.93      0.73      0.82       971
         2.0       0.69      0.76      0.72       970

    accuracy                           0.79      2912
   macro avg       0.80      0.79      0.79      2912
weighted avg       0.80      0.79      0.79      2912

[[854   0 117]
 [ 41 711 219]
 [179  57 734]]


{'accuracy': np.float64(0.79), 'f1': np.float64(0.79), 'mcc': np.float64(0.69)}

In [37]:
from xgboost import XGBClassifier

def xgb_objective(trial, X_train, Y_train, cv):
    xgb = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 10, 200),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        objective="multi:softprob",   # 多分類
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )
    scores = cross_validate(
        xgb,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "matthews_corrcoef"
        ],
        n_jobs=-1
    )
    
    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [38]:
# 建立 XGBClassifier 模型
study = optuna.create_study(direction="maximize")
study.optimize(
    lambda trial: xgb_objective(
        trial,
        training_X,
        training_Y,
        cv
    ),
    n_trials=200
)

best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_xgb = XGBClassifier(
    **study.best_params,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_xgb)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_snp_feature_xgb_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

xgb_fit = best_xgb.fit(training_X, training_Y)
xgb_score = xgb_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), xgb_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), xgb_score.astype(float))

[I 2026-11-18 11:54:16,001] A new study created in memory with name: no-name-175b85fa-25fc-4ef3-8a75-8a33c765021a


[I 2026-11-18 11:54:17,086] Trial 0 finished with value: 0.7108297371392529 and parameters: {'n_estimators': 129, 'max_depth': 4, 'learning_rate': 0.0017776961653655369, 'subsample': 0.8754227483880639, 'colsample_bytree': 0.6022167834581824, 'min_child_weight': 8, 'gamma': 2.09141676445646, 'reg_alpha': 4.8481168541326637e-08, 'reg_lambda': 1.3525413480291483e-06}. Best is trial 0 with value: 0.7108297371392529.
[I 2026-11-18 11:54:18,143] Trial 1 finished with value: 0.7585504711571047 and parameters: {'n_estimators': 151, 'max_depth': 4, 'learning_rate': 0.02799514725004295, 'subsample': 0.9802133424435194, 'colsample_bytree': 0.9358032122976001, 'min_child_weight': 9, 'gamma': 3.164707175236487, 'reg_alpha': 6.420038900594492e-05, 'reg_lambda': 0.00025028112640900385}. Best is trial 1 with value: 0.7585504711571047.
[I 2026-11-18 11:54:18,722] Trial 2 finished with value: 0.7355620059602117 and parameters: {'n_estimators': 11, 'max_depth': 5, 'learning_rate': 0.1202538429725464, 's

{'n_estimators': 162, 'max_depth': 9, 'learning_rate': 0.08416168999358555, 'subsample': 0.6304426458976082, 'colsample_bytree': 0.9818386963659661, 'min_child_weight': 4, 'gamma': 2.0118824739754766, 'reg_alpha': 3.1276597191705203e-06, 'reg_lambda': 0.0005457361101773588}
Best F1 Macro: 0.7967
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7965 ± 0.0032
precision_macro: 0.8020 ± 0.0042
recall_macro: 0.7965 ± 0.0030
f1_macro: 0.7967 ± 0.0032
matthews_corrcoef: 0.6968 ± 0.0048
              precision    recall  f1-score   support

         0.0       0.80      0.89      0.84       971
         1.0       0.90      0.75      0.82       971
         2.0       0.71      0.74      0.72       970

    accuracy                           0.80      2912
   macro avg       0.80      0.80      0.80      2912
weighted avg       0.80      0.80      0.80      2912

[[866   2 103]
 [ 46 731 194]
 [172  79 719]]


{'accuracy': np.float64(0.8), 'f1': np.float64(0.8), 'mcc': np.float64(0.7)}

In [39]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate
def hgbc_objective(trial, X_train, Y_train, cv):

    hgbc = HistGradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        max_iter=trial.suggest_int("max_iter", 10, 200),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 100),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 100),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-8, 10.0, log=True),
        max_bins=trial.suggest_int("max_bins", 64, 255),
        early_stopping=True,
        validation_fraction=0.2,
        n_iter_no_change=10,
        random_state=42
    )

    scores = cross_validate(
        hgbc,
        X_train,
        Y_train,
        cv = cv,
        scoring=[
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "matthews_corrcoef"
        ],
        n_jobs=-1
    )

    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [40]:
from sklearn.ensemble import HistGradientBoostingClassifier
study = optuna.create_study(direction="maximize")

study.optimize(
    lambda trial: hgbc_objective(trial, training_X, training_Y, cv),
    n_trials=200
)
best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_hgbc = HistGradientBoostingClassifier(
    **study.best_params,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=10,
    random_state=42
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_hgbc)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_snp_feature_hgbc_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

hgbc_fit = best_hgbc.fit(training_X, training_Y)
hgbc_score = hgbc_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), hgbc_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), hgbc_score.astype(float))

[I 2026-11-18 11:57:22,522] A new study created in memory with name: no-name-1f47fd08-21d5-49d9-ad30-1c99da4507d8
[I 2026-11-18 11:57:24,163] Trial 0 finished with value: 0.7928249672270218 and parameters: {'learning_rate': 0.022435099197744328, 'max_iter': 162, 'max_depth': 12, 'max_leaf_nodes': 27, 'min_samples_leaf': 44, 'l2_regularization': 0.01899760255231956, 'max_bins': 180}. Best is trial 0 with value: 0.7928249672270218.
[I 2026-11-18 11:57:25,266] Trial 1 finished with value: 0.7730535878263446 and parameters: {'learning_rate': 0.003944289574949264, 'max_iter': 67, 'max_depth': 11, 'max_leaf_nodes': 57, 'min_samples_leaf': 36, 'l2_regularization': 8.754427823322216e-06, 'max_bins': 158}. Best is trial 0 with value: 0.7928249672270218.
[I 2026-11-18 11:57:26,003] Trial 2 finished with value: 0.785294866896088 and parameters: {'learning_rate': 0.10853426645365533, 'max_iter': 155, 'max_depth': 6, 'max_leaf_nodes': 70, 'min_samples_leaf': 26, 'l2_regularization': 7.5687640148412

{'learning_rate': 0.08093480821190237, 'max_iter': 197, 'max_depth': 20, 'max_leaf_nodes': 17, 'min_samples_leaf': 52, 'l2_regularization': 5.674254336204832, 'max_bins': 170}
Best F1 Macro: 0.7964
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7963 ± 0.0042
precision_macro: 0.8023 ± 0.0040
recall_macro: 0.7962 ± 0.0046
f1_macro: 0.7964 ± 0.0040
matthews_corrcoef: 0.6966 ± 0.0064
              precision    recall  f1-score   support

         0.0       0.80      0.89      0.84       971
         1.0       0.90      0.75      0.82       971
         2.0       0.71      0.75      0.73       970

    accuracy                           0.80      2912
   macro avg       0.81      0.80      0.80      2912
weighted avg       0.81      0.80      0.80      2912

[[865   4 102]
 [ 46 732 193]
 [171  73 726]]


{'accuracy': np.float64(0.8), 'f1': np.float64(0.8), 'mcc': np.float64(0.7)}

In [41]:
import joblib

joblib.dump(svc, "sub_snp_feature_svm(3class).pkl")
joblib.dump(best_forest, "sub_snp_feature_rf(3class).pkl")
joblib.dump(best_xgb, "sub_snp_feature_xgboost(3class).pkl")
joblib.dump(best_hgbc, "sub_snp_feature_HGBC(3class).pkl")

['sub_snp_feature_HGBC(3class).pkl']

In [42]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    clf1 = joblib.load("sub_snp_feature_svc_fold_" + str(fold) + ".pkl")
    clf2 = joblib.load("sub_snp_feature_rf_fold_" + str(fold) + ".pkl")
    clf3 = joblib.load("sub_snp_feature_xgb_fold_" + str(fold) + ".pkl")
    clf4 = joblib.load("sub_snp_feature_hgbc_fold_" + str(fold) + ".pkl")

    # 建立新的模型
    model = VotingClassifier(
        estimators=[('svm', clf1), ('rf', clf2), ('xgb', clf3), ('hgbc', clf4)],
        voting='soft'  # 'soft' for probability-based voting
    )

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_feature_ensemble_fold_{fold}.pkl")
# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )    

clf1 = joblib.load("sub_snp_feature_svm(3class).pkl")
clf2 = joblib.load("sub_snp_feature_rf(3class).pkl")
clf3 = joblib.load("sub_snp_feature_xgboost(3class).pkl")
clf4 = joblib.load("sub_snp_feature_HGBC(3class).pkl")
ensemble = VotingClassifier(
    estimators=[('svm', clf1), ('rf', clf2), ('xgb', clf3), ('hgbc', clf4)],
    voting='soft'  # 'soft' for probability-based voting
)

# Train
ensemble.fit(training_X, training_Y)

# Predictions
y_pred = ensemble.predict(testing_X)

# Evaluate
print(classification_report(testing_Y.astype(float), y_pred.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), y_pred.astype(float))

Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.8000 ± 0.0039
precision_macro: 0.8064 ± 0.0043
recall_macro: 0.8000 ± 0.0038
f1_macro: 0.7999 ± 0.0038
matthews_corrcoef: 0.7027 ± 0.0058
              precision    recall  f1-score   support

         0.0       0.79      0.90      0.84       971
         1.0       0.92      0.76      0.83       971
         2.0       0.72      0.75      0.73       970

    accuracy                           0.80      2912
   macro avg       0.81      0.80      0.80      2912
weighted avg       0.81      0.80      0.80      2912

[[878   0  93]
 [ 49 738 184]
 [181  65 724]]


{'accuracy': np.float64(0.8), 'f1': np.float64(0.8), 'mcc': np.float64(0.71)}

In [43]:
joblib.dump(ensemble, "sub_snp_feature_ensemble(3class).pkl")

['sub_snp_feature_ensemble(3class).pkl']

In [47]:
model = joblib.load("./sub_snp_feature/sub_snp_feature_svm(3class).pkl")
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, 
    testing_X, 
    testing_Y,
    scoring="f1_macro",
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

feature_importance = pd.DataFrame({
    "feature": feature_list,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std
}).sort_values("importance_mean", ascending=False)

print(feature_importance)


                                             feature  importance_mean  \
1                                                AGE         0.156482   
4                                                 HB         0.087797   
7                                                BMI         0.074747   
3                                               SGPT         0.066812   
8                                        BODY_WEIGHT         0.066695   
6                                                HCT         0.065769   
5                                                 TG         0.065309   
2                                    FASTING_GLUCOSE         0.063975   
9  TPMI_Osteoporosis_osteopenia and pathological ...         0.046765   
0                                 MENSTRURATION_CURR         0.006832   

   importance_std  
1        0.007323  
4        0.003419  
7        0.005152  
3        0.003775  
8        0.004412  
6        0.002962  
5        0.003366  
2        0.003582  
9        0.00350